# Structured Outputs with Pydantic

This notebook compares a normal text response with data returned in a predictable Pydantic shape. It then reuses the same model to return details for multiple cars.

The example uses an OpenAI chat model. Load a valid `OPENAI_API_KEY` from your local `.env` file before running the model cells, and keep secrets out of source control.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

# This model first returns ordinary text; the same client will later be
# configured to return data that matches a Pydantic model.
llm = ChatOpenAI(
    model="gpt-6-luna",
)

## 1. Start with a Plain-Text Response

The input contains four specific facts. First ask for them without a schema so you can see the difference between a readable answer and fields your code can reliably access.

In [3]:
text = "The Toyota GR Yaris runs on a 1618 cc engine that delivers 268 horsepower."
unstructured_response = llm.invoke(
    f"Extract the make, model, engine capacity in cc, and horsepower from this description:\n{text}"
)

In [4]:
# A standard chat response is text, so a field like .make is not available here.
print(unstructured_response.content)

- **Make:** Toyota
- **Model:** GR Yaris
- **Engine capacity:** 1618 cc
- **Horsepower:** 268 hp


## 2. Define the Shape You Need

`Car` describes the fields the application expects. `Field` descriptions clarify each value, while the Python types make the expected format explicit. Structured output helps enforce this shape; it does not independently verify that the model's facts are true.

In [5]:
from pydantic import BaseModel, Field

# Pydantic turns the expected response into a reusable, typed data model.
class Car(BaseModel):
    make: str = Field(description="Manufacturer of vehicle")
    model: str = Field(description="Model of vehicle")
    engine_cc: int = Field(description="Engine capacity in cubic centimeters")
    horsepower: int = Field(description="Total horsepower produced by the engine")

## 3. Ask for Structured Output

`with_structured_output(Car)` connects the model to this schema. The model still interprets the request, but LangChain returns the result as a `Car` object instead of an unstructured text paragraph.

In [6]:
# Configure the model to return data matching the Car schema.
structured_llm = llm.with_structured_output(Car)

In [7]:
# Ask the same extraction question, now expecting a typed Car object.
structured_response = structured_llm.invoke(
    f"Extract the make, model, engine capacity in cc, and horsepower from this description:\n{text}"
)

In [8]:
# The result is a Car instance: inspect the object, convert it to a dict,
# or access one field directly for use in application code.
print(structured_response)
print(structured_response.model_dump())
print(structured_response.make)
print(structured_response.model)
print(structured_response.engine_cc)
print(structured_response.horsepower)

make='Toyota' model='GR Yaris' engine_cc=1618 horsepower=268
{'make': 'Toyota', 'model': 'GR Yaris', 'engine_cc': 1618, 'horsepower': 268}
Toyota
GR Yaris
1618
268


## 4. Return a List of Structured Objects

For multiple results, wrap the existing `Car` model in a response model with a list field. This keeps each car's fields consistent while allowing the number of cars to vary.

In [9]:
from typing import List

# Reuse Car so every item in the returned list has the same fields.
class InfoAboutMultipleCars(BaseModel):
    cars: List[Car]

In [10]:
# Use the list schema to request several cars with the same set of details.
cars_llm = llm.with_structured_output(InfoAboutMultipleCars)
cars_response = cars_llm.invoke(
    "Return details for three well-known cars: Toyota GR Yaris, Honda Civic Type R, and Ford Mustang GT. "
    "For each, provide the make, model, engine capacity in cc, and horsepower."
)

In [11]:
# Access the list of validated Car objects returned by the response model.
cars_response.cars

[Car(make='Toyota', model='GR Yaris', engine_cc=1618, horsepower=300),
 Car(make='Honda', model='Civic Type R', engine_cc=1996, horsepower=315),
 Car(make='Ford', model='Mustang GT', engine_cc=5038, horsepower=480)]

In [12]:
# Each list item is a Car object, so its fields are directly accessible.
cars_response.cars[1].model

'Civic Type R'

## Why Use Structured Output?

A predictable response is easier to pass into application code, serialize, store, or display. Pydantic helps enforce the expected fields and types, but it does not confirm that the model's values are factually correct. Verify important data against a trusted source.